# Item-Item Collaborative Filtering Recommender
## Spotify MPD

This notebook implements an item-item collaborative filtering recommender.
For each track in the seed playlist, we find the tracks most frequently
co-occurring with it across all training playlists. Scores are aggregated
across all seed tracks and the top-K candidates are recommended.

No training is required — the model is built purely from co-occurrence
counts in the training split produced by `preprocess.py`.

**Prerequisites:** `src/ingest.py` and `src/preprocess.py` must have been run first.

---
## 0. Setup

In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROCESSED = Path('../processed')
OUT       = Path('outputs')
OUT.mkdir(exist_ok=True)

SEED_RATIO  = 0.8
K_VALUES    = [1, 5, 10, 20]
MAX_EVAL    = 5000
RANDOM_SEED = 42
TOP_N_COOC  = 100   # for each seed track, keep top-N co-occurring tracks
MIN_COOC    = 3     # minimum co-occurrence count to include a pair

np.random.seed(RANDOM_SEED)

---
## 1. Load Data

In [2]:
print('Loading data...')
vocab      = pd.read_parquet(PROCESSED / 'track_vocab.parquet')
train_seqs = pd.read_parquet(PROCESSED / 'train_seqs.parquet')
test_seqs  = pd.read_parquet(PROCESSED / 'test_seqs.parquet')

# vocab index <-> URI mappings
idx_to_uri = dict(zip(vocab['corpus_idx'], vocab['track_uri']))
uri_to_idx = dict(zip(vocab['track_uri'], vocab['corpus_idx']))

print(f'  Vocab      : {len(vocab):,} tracks')
print(f'  Train seqs : {len(train_seqs):,}')
print(f'  Test seqs  : {len(test_seqs):,}')

Loading data...
  Vocab      : 2,262,292 tracks
  Train seqs : 799,998
  Test seqs  : 100,001


---
## 2. Build Co-occurrence Matrix

For every pair of tracks that appear in the same training playlist,
increment their co-occurrence count. We then build a lookup:
`track_idx -> [(co_track_idx, count), ...]` sorted by count descending.

Only pairs with at least `MIN_COOC` co-occurrences are kept to reduce
noise from rare tracks. For each track we keep only the top `TOP_N_COOC`
most frequent co-occurring tracks to keep memory usage manageable.

In [ ]:
print('Building co-occurrence counts...')
t0 = time.time()

cooc = defaultdict(lambda: defaultdict(int))

sequences = train_seqs['track_idxs'].tolist()
MAX_COOC_LEN = 50  # only use first 50 tracks per playlist for co-occurrence

for i, seq in enumerate(sequences):
    unique_tracks = list(dict.fromkeys(seq[:MAX_COOC_LEN]))
    for j in range(len(unique_tracks)):
        for k in range(j + 1, len(unique_tracks)):
            a, b = unique_tracks[j], unique_tracks[k]
            cooc[a][b] += 1
            cooc[b][a] += 1
    if (i + 1) % 100_000 == 0:
        print(f'  Processed {i+1:,} / {len(sequences):,} sequences  ({time.time()-t0:.1f}s)')
    
print(f'  Processed {i+1:,} / {len(sequences):,} sequences  ({time.time()-t0:.1f}s)')
print(f'\nRaw co-occurrence pairs: {sum(len(v) for v in cooc.values()):,}')

Building co-occurrence counts...
  Processed 100,000 / 799,998 sequences  (6.0s)
  Processed 200,000 / 799,998 sequences  (25.1s)
  Processed 300,000 / 799,998 sequences  (78.3s)
  Processed 400,000 / 799,998 sequences  (202.9s)
  Processed 500,000 / 799,998 sequences  (464.1s)
  Processed 600,000 / 799,998 sequences  (898.9s)
  Processed 700,000 / 799,998 sequences  (1477.6s)

Raw co-occurrence pairs: 526,576,062
Elapsed: 3470.7s


In [ ]:
print('Pruning and sorting co-occurrence table...')
t1 = time.time()

# For each track keep only top-N co-occurring tracks with min count
cooc_index = {}
for track_idx, neighbors in cooc.items():
    filtered = [(nb, cnt) for nb, cnt in neighbors.items() if cnt >= MIN_COOC]
    filtered.sort(key=lambda x: x[1], reverse=True)
    cooc_index[track_idx] = [nb for nb, _ in filtered[:TOP_N_COOC]]

# Free raw co-occurrence dict
del cooc

print(f'  Tracks in index : {len(cooc_index):,}')
print(f'  Elapsed         : {time.time()-t1:.1f}s')

Pruning and sorting co-occurrence table...


---
## 3. Recommender

Given a seed playlist, for each seed track look up its top co-occurring
tracks and add 1 to their score for each seed track they co-occur with.
Tracks already in the seed are excluded. The top-K by score are recommended.

In [ ]:
def recommend(seed_idxs: list[int], k: int) -> list[int]:
    """
    Given a list of seed track indices, return the top-k recommended
    track indices by aggregated co-occurrence score.
    """
    seen   = set(seed_idxs)
    scores = defaultdict(int)

    for track_idx in seed_idxs:
        for nb in cooc_index.get(track_idx, []):
            if nb not in seen:
                scores[nb] += 1

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [idx for idx, _ in ranked[:k]]

print('Recommender ready.')

### 3.1 Demo — single playlist

In [ ]:
sample = test_seqs.sample(1, random_state=RANDOM_SEED).iloc[0]
seq    = sample['track_idxs']
sp     = max(1, int(len(seq) * SEED_RATIO))
seed   = seq[:sp]
gt     = seq[sp:]

recs = recommend(seed, k=10)

# Decode to track names
meta = vocab.set_index('corpus_idx')[['track_uri']]
track_meta = pd.read_parquet(PROCESSED / 'track_meta.parquet',
                              columns=['track_uri', 'track_name', 'artist_name'])
uri_meta = track_meta.set_index('track_uri')

def decode(idx):
    uri = idx_to_uri.get(idx, '?')
    if uri in uri_meta.index:
        row = uri_meta.loc[uri]
        return f'{row["track_name"][:35]} – {row["artist_name"]}'
    return uri

print(f'Seed length : {len(seed)} tracks')
print(f'Holdout     : {len(gt)} tracks')
print(f'\nTop-10 recommendations:')
for i, idx in enumerate(recs, 1):
    hit = '✓' if idx in set(gt) else ' '
    print(f'  {i:2d}. [{hit}] {decode(idx)}')

---
## 4. Evaluation

For each test playlist:
- Seed = first 80% of tracks
- Holdout = remaining 20%
- Recommend top-K by co-occurrence score, compare with holdout

Metrics: Accuracy (Hit@1), Precision@K, Recall@K

In [ ]:
print(f'Evaluating on up to {MAX_EVAL:,} test playlists...')
t_eval = time.time()

rng        = np.random.default_rng(RANDOM_SEED)
seqs       = test_seqs['track_idxs'].tolist()
sample_idx = rng.choice(len(seqs), min(MAX_EVAL, len(seqs)), replace=False)

metrics  = {'Accuracy': []}
for k in K_VALUES:
    metrics[f'Prec@{k}']   = []
    metrics[f'Recall@{k}'] = []

skipped = 0
k_max   = max(K_VALUES)

for i, idx in enumerate(sample_idx):
    seq = seqs[idx]
    sp  = max(1, int(len(seq) * SEED_RATIO))
    if sp >= len(seq) - 1:
        skipped += 1
        continue

    seed    = seq[:sp]
    holdout = set(seq[sp:])
    if not holdout:
        skipped += 1
        continue

    recs = recommend(seed, k=k_max)

    metrics['Accuracy'].append(1 if recs and recs[0] in holdout else 0)
    for k in K_VALUES:
        hits = len(set(recs[:k]) & holdout)
        metrics[f'Prec@{k}'].append(hits / k)
        metrics[f'Recall@{k}'].append(hits / len(holdout))

    if (i + 1) % 1000 == 0:
        print(f'  [{i+1:>5}/{len(sample_idx)}]  '
              f'evaluated {i+1-skipped:,}  |  '
              f'skipped {skipped}  |  '
              f'{time.time()-t_eval:.1f}s')

results = {m: np.mean(v) for m, v in metrics.items() if v}
print(f'\nEvaluated {len(metrics["Accuracy"]):,} playlists  |  skipped {skipped}')

---
## 5. Results

In [ ]:
results_df = pd.DataFrame([results]).T.rename(columns={0: 'Score'})
print('Final Results:')
print(results_df.to_string(float_format='%.4f'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar([str(k) for k in K_VALUES],
            [results[f'Prec@{k}'] for k in K_VALUES], color='steelblue')
axes[0].set_title('Precision@K', fontweight='bold')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Score')

axes[1].bar([str(k) for k in K_VALUES],
            [results[f'Recall@{k}'] for k in K_VALUES], color='coral')
axes[1].set_title('Recall@K', fontweight='bold')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Score')

plt.suptitle('Item-Item Collaborative Filtering', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'itemCF_results.png', bbox_inches='tight')
plt.show()

---
## 6. Comparison with Other Models

| Model | Accuracy | Prec@1 | Prec@10 | Recall@10 | Recall@20 |
|---|---|---|---|---|---|
| GRU | 0.0612 | 0.1178 | 0.0720 | 0.0828 | 0.1273 |
| Transformer | 0.0604 | 0.1074 | 0.0681 | 0.0786 | 0.1230 |
| Genre Recommender | — | 0.0492 | 0.0386 | 0.0185 | 0.0309 |
| Content-Based Audio | — | 0.0002 | 0.0001 | 0.0002 | 0.0003 |
| **Item-Item CF (this notebook)** | *see above* | *see above* | *see above* | *see above* | *see above* |

---
## 7. Summary

Item-item collaborative filtering recommends tracks based purely on how often
they co-occur with seed tracks across all training playlists. It requires no
training, no audio features, and no genre labels — only the playlist structure itself.

**How it differs from previous approaches:**

| Approach | Signal used | Training needed |
|---|---|---|
| GRU / Transformer | Sequential co-occurrence (ordered) | Yes — 15 epochs |
| Genre Recommender | Playlist name keywords + popularity | No |
| Content-Based Audio | Acoustic feature similarity | No |
| **Item-Item CF** | **Unordered co-occurrence counts** | **No** |

The key difference from the GRU/Transformer is that item-item CF treats the
playlist as an unordered bag of tracks — it does not use position information.
The GRU/Transformer exploit the sequential order of tracks, which is additional
signal that item-item CF ignores.